### RAG pipelines - data ingestion to vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter   
from pathlib import Path


/tmp/ipykernel_31144/3675097828.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader


In [2]:
### read all the pdfs insdie the directory
def process_all_pdf(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # find all pdfs
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    
    print(f'found {len(pdf_files)} PDF files to process')
    
    for pdf_file in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # add extra meta data 
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print('loaded',len(documents))
        except Exception as e:
            print('error',e)
    print('\n Total doc loaded' , len(all_documents))
    return all_documents
        
    
   

In [3]:
all_pdf_documents = process_all_pdf('../data/pdf') 

found 4 PDF files to process
loaded 1
loaded 19
loaded 11
loaded 2

 Total doc loaded 33


In [4]:
def split_documents(documents , chunk_size = 1000 , chunk_overlap = 200):
    #lap is both docs contian same chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ['\n\n' , '\n' , " " , ""]   
    )
    
    split_docs = text_splitter.split_documents(documents)
    
    print(f'split{len(documents)} documents into {len(split_docs)} chunks')
    
    if split_docs:
        print(f'example chunk')
        print(f'content : {split_docs[0].page_content[:200]}.....')
        print(f'metadata : {split_docs[0].metadata}')
    return split_docs
        

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

split33 documents into 67 chunks
example chunk
content : Contact
23bq1a05a9@vvit.net
www.linkedin.com/in/sudhakar-
reddy-konda-769320299 (LinkedIn)
Top Skills
Hugging Face Products
Decision-Making
Data Analysis
Sudhakar Reddy konda
Final Year CSE @ VVIT | A.....
metadata : {'producer': 'Apache FOP Version 2.3', 'creator': 'PyPDF', 'creationdate': '2026-09-27T06:31:51+00:00', 'title': 'Resume', 'author': 'LinkedIn', 'subject': 'Resume generated from profile', 'source': '../data/pdf/a9_lnProfile.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'a9_lnProfile.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Apache FOP Version 2.3', 'creator': 'PyPDF', 'creationdate': '2026-09-27T06:31:51+00:00', 'title': 'Resume', 'author': 'LinkedIn', 'subject': 'Resume generated from profile', 'source': '../data/pdf/a9_lnProfile.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'a9_lnProfile.pdf', 'file_type': 'pdf'}, page_content='Contact\n23bq1a05a9@vvit.net\nwww.linkedin.com/in/sudhakar-\nreddy-konda-769320299 (LinkedIn)\nTop Skills\nHugging Face Products\nDecision-Making\nData Analysis\nSudhakar Reddy konda\nFinal Year CSE @ VVIT | AI & ML | FastAPI · LangGraph | B.Tech\n’27\nGuntur, Andhra Pradesh, India\nSummary\nFinal Year B.Tech CSE @ VVIT (CGPA 8.7).Building production-\ngrade AI systems — agentic , face recognition pipelines, and\nML-powered APIs — using Python, FastAPI, LangGraph, and\nReact.Open to AI/ML internships.\nEducation\nVasireddy Venkatadri International Technological University\nBachelor of Technology - BTech,\xa0Computer Science\

### Embeddings and vectorStoreDB

In [6]:
import numpy as np
import uuid
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import Tuple , Dict , List , Any
from sklearn.metrics.pairwise import cosine_similarity

/home/sudhakar-reddy/Public/naikRag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """ """
        self.model_name =  model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        
        try:
            """LOad the sentenceTransformer model"""
            
            print(f'loading embedding model {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'model loaded successfully. Embedding dimension {self.model.get_embedding_dimension()}')
        except Exception as e:
            print(f'Error  loading model{self.model_name} , {e}')
            raise
    
    def generate_embeddings(self , texts : List[str]) -> np.array :
        """generate embeddings for list of texts"""
        """encode method is used"""
        """ return : numpy array of embeddings with shape(len(texts) , embeddings)"""
        
        if not self.model:
            raise ValueError("model not loaded")

        print(f'generating embeddings ofr {len(texts)} texts...')
        embeddings = self.model.encode(texts , show_progress_bar = True)
        print(f'generated embeddings with shpae {embeddings.shape}')
        
        return embeddings
        
        
embedding_manager = EmbeddingManager()
embedding_manager

        
        
        
        

loading embedding model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1672.14it/s]


model loaded successfully. Embedding dimension 384


## vecotorStore 
#### manages document embeddings in a chromaDB vector store

In [8]:
class VectorStore:
    def __init__(self , collection_name : str = 'pdf_documents' , persist_directory : str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """ initialize chromadb client and collection """
        try:
            # create a persistent chromadb client
            os.makedirs(self.persist_directory , exist_ok = True)
            self.client = chromadb.PersistentClient(path  = self.persist_directory)
            
            # Get or create a collection
            self.collection = self.client.get_or_create_collection(
                                            name = self.collection_name,
                                            metadata = {'description' : 'PDF document embeddings for RAg'}
            )
            
            print(f'vector store initialised collection : {self.collection_name}')
            print(f'existing documents in collection: {self.collection.count()}')
        except Exception as e:
            print(f'error initialising vector store {e}')
            raise
        
    def add_documents(self , documents : List[Any] , embeddings : np.array):
        ''' add documents and their embeddings to a vector store '''
        """ args : 
        
        docu : list of lnaghcian docs
        embeddings : corresponding embeddings for the docus
        
        """
        if len(documents) != len(embeddings):
            raise ValueError("no of docs not matched with no of embeddings")
        print(f'adding docus to vector store {len(documents)}')
        
        # prepare data for chromdb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i , (doc , embedding) in enumerate(zip(documents,embeddings)):
            # generate unq id
            doc_id =f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] =  len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        # add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f'successfully added {len(documents)} doucs to vectorstore')
            print(f'total docs in collections {self.collection.count()}')
            
        except Exception as e:
            print('error addding docs in vector store , {e}')
            raise

vectorstore = VectorStore()
vectorstore
            
            
        


vector store initialised collection : pdf_documents
existing documents in collection: 268


In [9]:
texts = [doc.page_content for doc in chunks]
# convert the text to embedings
texts

['Contact\n23bq1a05a9@vvit.net\nwww.linkedin.com/in/sudhakar-\nreddy-konda-769320299 (LinkedIn)\nTop Skills\nHugging Face Products\nDecision-Making\nData Analysis\nSudhakar Reddy konda\nFinal Year CSE @ VVIT | AI & ML | FastAPI · LangGraph | B.Tech\n’27\nGuntur, Andhra Pradesh, India\nSummary\nFinal Year B.Tech CSE @ VVIT (CGPA 8.7).Building production-\ngrade AI systems — agentic , face recognition pipelines, and\nML-powered APIs — using Python, FastAPI, LangGraph, and\nReact.Open to AI/ML internships.\nEducation\nVasireddy Venkatadri International Technological University\nBachelor of Technology - BTech,\xa0Computer Science\xa0·\xa0(September\n2023\xa0-\xa02027)\nSri Chaitanya College of Education\n\xa0·\xa0(June 2021\xa0-\xa0March 2023)\n\xa0 Page 1 of 1',
 'Unit-1 \nIntroduction of IoT \n1. Overview \nIoT comprises things that have unique identities an d are connected to internet. By \n2020 there will be a total of 50 billion devices /things connected to internet. IoT is not limite

In [10]:
# generate the embeddings
enbeddings = embedding_manager.generate_embeddings(texts)



generating embeddings ofr 67 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.11it/s]

generated embeddings with shpae (67, 384)


In [11]:
# store in vector db
vectorstore.add_documents(chunks , enbeddings)

adding docus to vector store 67
successfully added 67 doucs to vectorstore
total docs in collections 335


## retriever pipeline from vectorstore

In [ ]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self,vector_store: VectorStore,embedding_manager: EmbeddingManager,):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query: str,top_k: int = 5,score_threshold: float = 0.0,) -> List[Dict[str, Any]]:
        try:
            # generate query embeddings
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            
            # search in vector store
            if hasattr(query_embedding, "tolist"):
                query_embedding = query_embedding.tolist()

            results = self.vector_store.collection.query(
                                                query_embeddings=[query_embedding],
                                                n_results=top_k,
                                                    )

            documents = (results.get("documents") or [[]])[0] or []
            metadatas = (results.get("metadatas") or [[]])[0] or []
            distances = (results.get("distances") or [[]])[0] or []
            ids = (results.get("ids") or [[]])[0] or []

            retrieved_docs = []
            for doc_id, document, metadata, distance in zip(
                ids, documents, metadatas, distances
            ):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                                        "id": doc_id,
                                        "content": document,
                                        "metadata": metadata,
                                        "similarity_score": similarity_score,
                                    })

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
        
    
    
    

In [13]:

rag_retriever.retrieve('M2M (Machine to Machine) ')

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.72it/s]

generated embeddings with shpae (1, 384)


[{'id': 'doc_8ab7c7ac_41',
  'content': 'Unit-2 \nIoT & M2M \n2. M2M (Machine to Machine) \nMachine-to-Machine (M2M) refers to networking of ma chines (or devices) for the \npurpose of remote monitoring and control and data e xchange .) Figure 1 shows the end-\nto-end architecture for M2M systems comprising of M2M area networks, communication \nnetwork and application domain).  An M2M area network comprises of machines (or M2M \nnodes) which have embedded hardware modules for sensing, actuation and \ncommunication.  Various communication protocols can be used for M2M local area networks \nsuch as ZigBee, Bluetooth, Modbus, M-Bus, Wireless M-Bus, P ower Line \nCommunication (PLC), etc. These communication protocols  provide connectivity between \nM2M nodes within an M2M area network. The communication network  provides \nconnectivity to remote M2M area networks. The commu nication Network can use either \nwired or wireless networks (IP-based).  While the M2M area networks use either',
 

In [14]:
rag_retriever.retrieve("Sudhakar Reddy konda")

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.00it/s]

generated embeddings with shpae (1, 384)


[]

In [15]:
rag_retriever.retrieve("Differences between M2M and IoT")

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.27it/s]

generated embeddings with shpae (1, 384)


[{'id': 'doc_e03855dc_45',
  'content': 'Fig 3: Communication Protocol \n\uf0d8 Machines in M2M vs Things in IoT:  The "Things" in loT refers to physical objects tha t \nhave unique identifiers and can Sense and communica te with their external environment \n(and user applications) or their internal physical states. The unique identifiers for the \nthings in loT are the IP addresses. (or MAC address es)) Things have software \ncomponents for accessing, processing, and storing s ensor information, or controlling \nactuators connected. IOT Systems can nave Heterogen eous things (e.g., a home \nautomation IoT system can include lol devices of va rious types, such as fire alarms, door \nalarms, lighting control devices, etc.M2M systems, in contrast to IoT. typically have \nhomogeneous machine types within an M2M area network \n\uf0d8 Hardware vs Software Emphasis : While the emphasis of M2M 1s more on hardware \nwith embedded modules, the emphasis of loT is more on software (loT devices ru

## Integration Vectordb context pipeline with LLM output

In [16]:
# simple RAG pipeline with GROQ llm

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

True

In [17]:
### intialize groq llm
groq_api_key = os.getenv('GROQ_API_KEY')


In [21]:

llm = ChatGroq(groq_api_key = groq_api_key , model_name="openai/gpt-oss-120b" , temperature=0.2 , max_tokens=1024)


In [ ]:
def rag_simple(query , retriever , llm , top_k =4):
    # retriever the context
    results = retriever.retrieve(query , top_k = top_k)
    # print('results ------> ',results)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    # print('context ---------->' ,context)
    if not context:
        return "no relevant context found to answer"

    # generate the answer using llm
    prompt = f"""use the following context to answer the question concisely.
                            context: {context}'
                            question : {query}
                answer : """
    response = llm.invoke([prompt.format(context = context , query = query)])
    # print(response)
    return response.content

    
    




In [39]:
answer = rag_simple(query="what is m2m" , retriever= rag_retriever , llm = llm)
answer

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.65it/s]

generated embeddings with shpae (1, 384)
results ------>  [{'id': 'doc_8ab7c7ac_41', 'content': 'Unit-2 \nIoT & M2M \n2. M2M (Machine to Machine) \nMachine-to-Machine (M2M) refers to networking of ma chines (or devices) for the \npurpose of remote monitoring and control and data e xchange .) Figure 1 shows the end-\nto-end architecture for M2M systems comprising of M2M area networks, communication \nnetwork and application domain).  An M2M area network comprises of machines (or M2M \nnodes) which have embedded hardware modules for sensing, actuation and \ncommunication.  Various communication protocols can be used for M2M local area networks \nsuch as ZigBee, Bluetooth, Modbus, M-Bus, Wireless M-Bus, P ower Line \nCommunication (PLC), etc. These communication protocols  provide connectivity between \nM2M nodes within an M2M area network. The communication network  provides \nconnectivity to remote M2M area networks. The commu nication Network can use either \nwired or wireless networks

'M2M (Machine‑to‑Machine) is the networking of machines or devices so they can monitor, control, and exchange data with each other remotely. It involves\u202f\n\n* **M2M area networks** – local links between devices (using protocols such as ZigBee, Bluetooth, Modbus, M‑Bus, Wireless M‑Bus, PLC, etc.), and  \n* **Communication networks** – wired or wireless IP‑based links that connect those local networks to distant sites,  \n\nenabling end‑to‑end remote sensing, actuation, and data communication.'

In [33]:
answer = rag_simple(query="what is java 25" , retriever= rag_retriever , llm = llm)
answer

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.98it/s]

generated embeddings with shpae (1, 384)


'no relevant context found to answer'

In [86]:

def rag_advance(query , retriever , llm , top_k = 5 , min_score = 0.3 , return_context = False):
    """  RAG pipeline with extra features
        - Return answer, sources, confidence score, and optionally full context.
      """
    results = retriever.retrieve(query ,top_k = top_k , score_threshold = min_score )

    if not results:
        return {'answer' :' no relevant context found '  , 'sources' : [] , 'confidence' : 0.0 , 'context' : ''}
    
    # prepare context and sources

    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source' : doc['metadata'].get('source_file' ,doc['metadata'].get('source' , 'unknown') ) ,
        'page' : doc['metadata'].get('page' , 'unknown'),
        'score'  : doc['similarity_score'],
        'preview' : doc['content'][:320]+ '....'
        
    } for doc in results ]

    # confidence = max([doc['similarity_score'] for doc in results ])



    # generate ans
    prompt = f""" use the following context to answer the question concisely .\n context : \n {context} \n question: {query} \n\n answer:"""
    # response = llm.invoke([prompt.format(context = context , query = query)])
    
    # streaming the output 
    for chunk in llm.stream(prompt):
        if chunk.content:
            yield chunk.content

    # output = {
    #     'answer' : response.content , 'sources' : sources 
    # }

    # if return_context :
    #     output['context'] = context

    # return output





In [ ]:
for chunk in rag_advance(query = "Differences between M2M and IoT", retriever = rag_retriever, llm = llm):
    print(chunk, end="", flush=True)
    # output is displayed chunk after chunk ,not wait unitl entire respone from llm

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.78it/s]

generated embeddings with shpae (1, 384)


**Key differences between M2 M and IoT**

| Aspect | M2 M | IoT |
|--------|------|-----|
| **Entity** | “Machines” (often similar devices) | “Things” – physical objects with unique IDs |
| **Identification** | Typically device‑specific IDs, not always IP‑based | IP (or MAC) addresses give each thing a global network identity |
| **Network composition** | Generally **homogeneous** – same type of machines in an area network | **Heterogeneous** – many different device types (sensors, actuators, appliances, etc.) |
| **Emphasis** | **Hardware‑centric** – embedded modules, low‑level control | **Software‑centric** – on‑device processing, data handling, cloud integration |
| **Functionality** | Focus on direct machine‑to‑machine communication | Enables sensing, data collection, processing, and interaction with user applications and external environments |
| **Typical use‑case** | Industrial control loops, SCADA, telemetry | Smart homes, smart cities, wearables, connected consumer products |


In [ ]:
q1 = rag_advance("Differences between M2M and IoT" ,rag_retriever , llm , top_k=5 , min_score=0.1, return_context=True)
print("answer ------>" , q1['answer'])
print('sources ----->' , q1['sources'])
# print('confidence' , q1['confidence'])
print('context preview  ----->' , q1['context'][:300])

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.78it/s]

generated embeddings with shpae (1, 384)


answer ------> **Key differences between M2 M and IoT**

| Aspect | M2 M | IoT |
|--------|------|-----|
| **Entity** | “Machines” – typically homogeneous devices within a closed area network. | “Things” – physical objects with unique IDs (IP or MAC) that can sense, communicate, and act. |
| **Identifiers** | Often proprietary or device‑specific IDs. | Standard network IDs (IP/MAC) enable direct addressing on the Internet. |
| **Device diversity** | Mostly the same type of hardware (e.g., a fleet of identical sensors). | Heterogeneous mix of devices (sensors, actuators, appliances, wearables, etc.). |
| **Emphasis** | Hardware‑centric: embedded modules, low‑level protocols. | Software‑centric: middleware, data processing, cloud services, APIs. |
| **Network scope** | Limited, often isolated or private networks. | Open, IP‑based networks that can interoperate globally. |
| **Typical use‑case** | Industrial automation, SCADA, machine‑to‑machine telemetry. | Smart homes, smart cities, wea

In [85]:
q1 = rag_advance("Differences between iphone and samsung" ,rag_retriever , llm , top_k=5 , min_score=0.1, return_context=True)
print("answer ------>" , q1['answer'])
print('sources ----->' , q1['sources'])
# print('confidence' , q1['confidence'])
print('context preview  ----->' , q1['context'][:300])

generating embeddings ofr 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.02it/s]

generated embeddings with shpae (1, 384)
answer ------>  no relevant context found 
sources -----> []
context preview  -----> 


In [ ]:
# QAllm =  ChatGroq(groq_api_key = groq_api_key , model_name="openai/gpt-oss-120b" , temperature=0.1 , max_tokens=524)


In [ ]:
# QAllm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond appropriately. No special instructions. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.080981429, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.002872392, 'prompt_tokens_details': None, 'queue_time': 0.337866916, 'total_time': 0.083853821}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a133f7-2b32-7763-9cab-0816b926a4c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [ ]:
# QAllm.invoke('what is attention all you need')

AIMessage(content='**“Attention\u202fIs\u202fAll\u202fYou\u202fNeed”** is the title of a landmark research paper (Vaswani\u202fet\u202fal., 2017) that introduced the **Transformer** architecture—a model that has become the foundation for most modern natural‑language‑processing (NLP) systems, including BERT, GPT, T5, and many others.\n\nBelow is a quick‑look at what the paper proposes, why it matters, and how the core ideas work.\n\n---\n\n## 1. The Problem It Solves\nBefore 2017, the dominant sequence‑to‑sequence models for tasks like translation were **RNNs** (LSTMs/GRUs) with an **attention** mechanism.  \n* RNNs process tokens one‑by‑one, which makes training slow (hard to parallelize) and limits the distance over which information can flow.  \n* Adding attention helped the decoder focus on relevant encoder states, but the underlying recurrence still bottlenecked speed and memory.\n\nThe authors asked: *Can we get rid of recurrence entirely and still capture the relationships betwee